In [8]:
import pyreadr
import pandas as pd
import xml
import re
from bs4 import BeautifulSoup
import string
raw_claims = pyreadr.read_r('../../data/claims-raw.RData')
raw_claims_df = raw_claims['claims_raw']
raw_claims_df['text_tmp'].head(5)


0    <!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 S...
1    <!DOCTYPE html>\n<html lang="en">\n<head>\n   ...
2    <!DOCTYPE html>\n<html>\n<head>\n<meta charset...
3    <!DOCTYPE HTML>\n<html lang="en">\n<head>\n\t<...
4    <!DOCTYPE html>\n<html>\n<head>\n<meta charset...
Name: text_tmp, dtype: str

In [ ]:
'''
def parse_fn(html):
    soup = BeautifulSoup(html, 'html.parser')
    clean_text = soup.get_text().translate(str.maketrans('', '', string.punctuation)).replace("\n", "")
    print(clean_text)
'''
def parse_fn(html):
    soup = BeautifulSoup(html, features = "xml")
    try:
        for script in soup(["script", "style"]):
            script.decompose() # destroy script and style elements

        # get head and paragraph text
        ph_text = ' '.join(spoon.get_text() for spoon in soup.select('h1, p'))

        # remove urls and emails
            # \S@\S+ means non-whitespace characters followed by @ then followed by 1 or more non-whitespace characters
            # pattern = r'https?://\S+|www\.\S+' which means http with optional s followed by :// and then 1 or more non-whitespace characters
            # |www\.\S+ means or www. then 1 or more non-whitespace characters
        text = re.sub(r'\S@\S+|https?://\S+', "", ph_text)

        # remove apostraphes

        text = re.sub("'", "",text)

        # remove non-alphabetic characters and nbsp
        text = re.sub(r'[^a-zA-Z\s]', ' ', text)

        text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text) # add space between camel case words

        text = text.lower()

        text = re.sub(r'\s+', ' ', text).strip() # collapse multiple spaces into one and strip leading/trailing whitespace
        # remove punctuation and newlines
        
        return text 
    except Exception as err:
        print(f"Error parsing HTML: {err}")
        return None


def parse_data(df):
    df = df[df['text_tmp'].str.contains('<!', regex = False)].copy() # filter for rows that contain html tags
    df['text_clean'] = df['text_tmp'].apply(parse_fn) # apply parse_fn and create a new column
    return df

clean_claims_df = parse_data(raw_claims_df)


In [11]:
clean_claims_df['text_clean'].head(5)


0                                                     
1    the following official arrest record for jonat...
2    new mugshots in miami dade county fl friday ja...
3    the information on this website is taken from ...
4    clayton thomas arrest details clayton thomas a...
Name: text_clean, dtype: str